# Lab: Build a Winter Storm Summary Pipeline

In the demo we built an ETL pipeline that aggregated weather data to monthly averages. In this lab you'll tackle a different problem using the same dataset: **build a per-station winter storm summary** by joining two tables, filtering to storm days, and aggregating.

**Tips:**
- Use the **Databricks Assistant** (cmd/ctrl + I) to help you write code
- The source tables live in `{catalog}.{schema}` — use the widgets below
- Each exercise describes *what* to do — it's up to you *how* to do it

## Setup

Run the two cells below to configure your notebook.

In [ ]:
current_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
unique_table_suffix = current_user.split('@')[0].replace('.', '_')

In [ ]:
dbutils.widgets.text("catalog", "classic_stable_been2c_catalog", "Catalog")
dbutils.widgets.text("schema", "weather", "Schema")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print("catalog:", catalog)
print("schema:", schema)

---
## Exercise 1: Load & Join

The setup script created two normalized tables:

| Table | Contents |
|---|---|
| `station_location` | One row per station — `station_code`, postal code, country, lat, long |
| `daily_weather_measurement` | One row per station per day — all weather observations |

Load both tables into DataFrames and **join them on `station_code`** so each measurement row has its station's latitude and longitude.

In [ ]:
# Load station_location and daily_weather_measurement
# Join them on station_code
# Display a sample to verify the join worked


---
## Exercise 2: Filter to Storm Days

From the joined DataFrame, keep only rows where **snow or ice occurred** (`has_snow = 'True'` or `has_ice = 'True'`).

Display a few rows and a count so you can see how much data you're working with.

In [ ]:
# Filter to rows where has_snow = 'True' OR has_ice = 'True'
# Display a sample and print the row count


---
## Exercise 3: Aggregate a Storm Summary

Group by `station_code` and compute a storm summary with these columns:

| Output column | Aggregation |
|---|---|
| `storm_days` | count of rows |
| `total_snowfall` | sum of `snow_total` |
| `max_wind_gust` | max of `wind_gust_max` |
| `avg_temperature` | average of `temperature_avg` |

Order the result by `total_snowfall` descending.

In [ ]:
# Group by station_code
# Compute: storm_days (count), total_snowfall (sum), max_wind_gust (max), avg_temperature (avg)
# Order by total_snowfall descending
# Display the result


---
## Exercise 4: Save as a Delta Table

Write your storm summary DataFrame as a Delta table named:

`{catalog}.{schema}.storm_summary_{unique_table_suffix}`

Then read it back and display it to confirm.

In [ ]:
# Write the storm summary as a Delta table (overwrite mode)
# Read it back and display to verify


---
## Challenge: Filter by Proximity with Geospatial SQL

Databricks has built-in [geospatial functions](https://docs.databricks.com/en/sql/language-manual/sql-ref-geospatial-functions.html) that work directly in SQL and PySpark.

Go back to your joined DataFrame (before the storm filter) and use `ST_DISTANCE` and `ST_POINT` to **keep only stations within 100 km of Raleigh, NC** (latitude `35.7796`, longitude `-78.6382`).

Then re-run your storm filter and aggregation on just those nearby stations and save as `{catalog}.{schema}.storm_summary_raleigh_{unique_table_suffix}`.

Useful functions:
- `ST_POINT(longitude, latitude)` — creates a geometry point
- `ST_DISTANCE(point1, point2)` — returns distance in meters

In [ ]:
# Filter your joined DataFrame to stations within 100 km of Raleigh, NC
# Raleigh: latitude = 35.7796, longitude = -78.6382
# Then apply the storm filter and aggregation from Exercises 2-3
# Save as storm_summary_raleigh_{unique_table_suffix}


In [ ]:
# Save as a Delta table and display
# How do the results compare to the full-country summary?


---
**Nice work!** You've built a pipeline that joins, filters, aggregates, and persists weather data — and optionally added geospatial proximity filtering. This notebook is ready to be scheduled as a Databricks Job.